# Card Detector - Colab GPU training (v2, disconnect-safe)

**Run all cells top to bottom in one sitting** (don't leave it idle - free Colab recycles the runtime and wipes everything).

First: `Runtime -> Change runtime type -> T4 GPU`.

The final cell trains, evaluates on the held-out test auction, and copies `best.pt` to Drive immediately, so a later disconnect can't lose it.

In [ ]:
# 1. Confirm GPU (should show Tesla T4)
!nvidia-smi

In [ ]:
# 2. Mount Drive, install Ultralytics, set config
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

DRIVE_DIR = '/content/drive/MyDrive/card_detector'
RUN_NAME  = 'card_v1'
EPOCHS    = 60
IMGSZ     = 640
BATCH     = 16

In [ ]:
# 3. Unzip dataset, locate data.yaml, repoint its path to Colab
import glob, shutil
from pathlib import Path

shutil.rmtree('/content/ds', ignore_errors=True)
!unzip -q "{DRIVE_DIR}/dataset.zip" -d /content/ds

yamls = glob.glob('/content/ds/**/data.yaml', recursive=True)
assert yamls, 'data.yaml not found - did you zip the dataset/ folder?'
DATA_YAML = yamls[0]
ds_root = str(Path(DATA_YAML).parent)
lines = Path(DATA_YAML).read_text().splitlines()
lines = [f'path: {ds_root}' if l.startswith('path:') else l for l in lines]
Path(DATA_YAML).write_text('\n'.join(lines) + '\n')
print('DATA_YAML =', DATA_YAML)
for sp in ('train','val','test'):
    print(sp, len(glob.glob(f'{ds_root}/images/{sp}/*')), 'images')

In [ ]:
# 4. Train + evaluate on val AND test + save to Drive - all in one cell (Python API)
import os, shutil
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
            patience=20, device=0, name=RUN_NAME, seed=0, plots=True)

save_dir = str(model.trainer.save_dir)        # actual run dir (handles name bumps)
best_pt  = f'{save_dir}/weights/best.pt'

best = YOLO(best_pt)
mv = best.val(data=DATA_YAML, split='val',  device=0)
mt = best.val(data=DATA_YAML, split='test', device=0)
print('\n=================== SCORES ===================')
print(f'VAL  (action06): mAP50={mv.box.map50:.3f}  P={mv.box.mp:.3f}  R={mv.box.mr:.3f}  mAP50-95={mv.box.map:.3f}')
print(f'TEST (action05): mAP50={mt.box.map50:.3f}  P={mt.box.mp:.3f}  R={mt.box.mr:.3f}  mAP50-95={mt.box.map:.3f}')

out = f'{DRIVE_DIR}/output_{RUN_NAME}'
os.makedirs(out, exist_ok=True)
shutil.copy(best_pt, f'{out}/best.pt')
shutil.copytree(save_dir, f'{out}/run', dirs_exist_ok=True)
print('\nSaved best.pt + run to', out, '- safe from disconnects now.')